# Financials View — Preparing the Financial History Table


This notebook prepares the **financial history dataset** to be used as an input signal in the thesis pipeline.  
The goal is to create a consistent and analysis-ready financial view that can be safely merged with supplier and contract data.

Financial risk is treated as one of the structured risk signals in the decision-support framework, complementing operational and contractual indicators.

**Data Characteristics**

The raw financial dataset contains **multiple records for the same company and closing year**.  
These repeated rows reflect different technical assessment snapshots captured over time, identified by the timestamp:

- `created_at_utc`

This means that a company may have several financial records for the same year, representing updates or revisions to the financial assessment.


**Structure for the Pipeline**

For the thesis data model, the financial dataset must follow the structure:

**one row = one company in one year**

This structure ensures:

- consistent time alignment with contract-year data  
- stable joins across datasets  
- reproducible modeling and aggregation  
- interpretable financial risk signals  


To achieve the required structure, the dataset is cleaned by keeping **only the latest available record** for each combination of:

- `moodys_bvd_id`
- `closing_year`

The latest record is determined using:

- `created_at_utc` (most recent timestamp)

This step removes duplicate technical snapshots while preserving the full yearly financial history.


Resulting Dataset

After cleaning:

- each company has **at most one record per year**
- the financial history remains intact
- duplicate technical updates are removed
- the dataset becomes suitable for time-based joins

This cleaned dataset is referred to as:

**`financials_clean`**

**Key Variable**

The primary variable used in later analysis and modeling is:

**`Financial_level`**

This variable represents the financial risk classification provided by the external financial source (moodys).

Typical categories include:

- Low financial risk  
- Medium financial risk  
- High financial risk  
- Missing (no available financial assessment)

Missing financial risk values are expected, as not all suppliers are covered by external financial rating systems.

**Coverage Baseline**

The cleaned financial dataset provides the baseline availability of financial risk information before merging with contracts.

Example:

- Total records: 3,529  
- Records with financial risk: 1,635  
- Coverage: **46.3%**

This baseline will be compared to coverage after merging with the contract dataset.

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
src_path = project_root / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("project_root:", project_root)
print("src_path:", src_path)

project_root: /Users/annita/Desktop/Thesis/Signal_Fusion_with_Meta-learners-
src_path: /Users/annita/Desktop/Thesis/Signal_Fusion_with_Meta-learners-/src


In [2]:
import pandas as pd

In [3]:
df_financials = pd.read_csv("/Users/annita/Desktop/Thesis/tables/financials/moodys_financials.csv")

In [4]:
print("Shape:")
print(df_financials.shape)

print("\nColumns:")
print(df_financials.columns.tolist())

print("\nData types:")
print(df_financials.dtypes)

print("\nSample:")
df_financials.head()

Shape:
(17390, 52)

Columns:
['moodys_bvd_id', 'moodys_company_name', 'Status', 'Implied_rating - original', 'Implied_rating', 'risk_level', 'output_text', 'Financial_level', 'closing_year', 'closing_date', 'Number_of_months', 'Net_Salesth_DKK', 'Operating_revenue_Turnover_th_DKK', 'Gross_profit_th_DKK', 'EBIT_th_DKK', 'Net_income_th_DKK', 'Interest_paid_th_DKK', 'EBITDA_th_DKK', 'Costs_of_goods_soldth_DKK', 'Total_assets_th_DKK', 'Non-current_assetsth_DKK', 'Current_assets_th_DKK', 'Cash_and_cash_equivalent_th_DKK', "Total_shareholders'_funds_and_liabilitiesth_DKK", 'Long_term_debtth_DKK', 'Other_current_assetsth_DKK', 'Shareholders_funds_th_DKK', 'Current_liabilities_th_DKK', 'Non_current_liabilities_th_DKK', 'Other_non_current_liabilitiesth_DKK', 'Other_current_liabilitiesth_DKK', 'Working_capital_th_DKK', 'Profit_margin', 'Gross_margin', 'EBITDA_margin', 'EBIT_margin', 'Current_ratio', 'Gearing', 'Cash_ratio', 'Long_term_Gearing', 'Short_term_Gearing', 'Total_liabilities_Equity_rat

,moodys_bvd_id,moodys_company_name,Status,Implied_rating - original,Implied_rating,risk_level,output_text,Financial_level,closing_year,closing_date,...,Long_term_liabilities_Equity_ratio,Short_term_liabilities_Equity_ratio,Interest_coverage_ratio,Solvency_ratio_Asset_based,Debt_Asset_ratio,ROE_using_Net_income,ROA_using_Net_income,Net_assets_turnover,Number_of_employees,created_at_utc
0,GB03898252,THE Turner Agency Limited,Active,Ba3,Ba,Take caution,Use caution. Input from outdated and/or partia...,NaN,2023.0,2023-12-31,...,1.773269,226.829799,NaN,30.432,0.000000,NaN,NaN,NaN,31.0,2026-03-12 11:47:28.083189+00:00
1,FR900421140,Kliper,Active,B2,B,Do not source,Sourcing exclusion recommended. Input from out...,NaN,2022.0,2022-12-31,...,-10508.485250,-16104.342062,NaN,-0.377,0.468776,NaN,NaN,NaN,NaN,2026-03-12 11:47:28.083189+00:00
2,GB11779532,67 Health Limited,Active,Ba1,Ba,Take caution,Use caution. Input from outdated and/or partia...,NaN,2022.0,2022-12-31,...,0.115574,88.746533,NaN,52.949,0.038929,NaN,NaN,NaN,9.0,2026-03-12 11:47:28.083189+00:00
3,GB11429590,Caristo Diagnostics Limited,Active,Baa3,Baa,Take caution,Use caution. Input from outdated and/or partia...,NaN,2022.0,2022-12-31,...,0.000000,35.170227,NaN,73.981,NaN,NaN,NaN,NaN,33.0,2026-03-12 11:47:28.083189+00:00
4,MXCTS160922QLA,Corporate Travel Services Worldwide,Active,B1,B,Do not source,Sourcing exclusion recommended. Mitigation act...,High financial risk,2025.0,2025-03-31,...,14.500631,126.422891,NaN,41.507,0.000000,1.412,2.344,3.641,552.0,2026-03-12 11:47:28.083189+00:00


`one row = one Moody’s company in one closing year`

In [5]:
dup = df_financials.duplicated(
    ["moodys_bvd_id", "closing_year"]
).sum()

print("Duplicate company-year rows:", dup)

Duplicate company-year rows: 12996


In [6]:
print(
    "Unique companies:",
    df_financials["moodys_bvd_id"].nunique()
)

Unique companies: 1989


In [7]:
print(
    df_financials["closing_year"]
    .sort_values()
    .unique()
)

[1997. 2000. 2001. 2002. 2003. 2004. 2005. 2006. 2007. 2008. 2009. 2010.
 2011. 2012. 2013. 2014. 2015. 2016. 2017. 2018. 2019. 2020. 2021. 2022.
 2023. 2024. 2025. 2026.   nan]


In [8]:
df_financials.head()

,moodys_bvd_id,moodys_company_name,Status,Implied_rating - original,Implied_rating,risk_level,output_text,Financial_level,closing_year,closing_date,...,Long_term_liabilities_Equity_ratio,Short_term_liabilities_Equity_ratio,Interest_coverage_ratio,Solvency_ratio_Asset_based,Debt_Asset_ratio,ROE_using_Net_income,ROA_using_Net_income,Net_assets_turnover,Number_of_employees,created_at_utc
0,GB03898252,THE Turner Agency Limited,Active,Ba3,Ba,Take caution,Use caution. Input from outdated and/or partia...,NaN,2023.0,2023-12-31,...,1.773269,226.829799,NaN,30.432,0.000000,NaN,NaN,NaN,31.0,2026-03-12 11:47:28.083189+00:00
1,FR900421140,Kliper,Active,B2,B,Do not source,Sourcing exclusion recommended. Input from out...,NaN,2022.0,2022-12-31,...,-10508.485250,-16104.342062,NaN,-0.377,0.468776,NaN,NaN,NaN,NaN,2026-03-12 11:47:28.083189+00:00
2,GB11779532,67 Health Limited,Active,Ba1,Ba,Take caution,Use caution. Input from outdated and/or partia...,NaN,2022.0,2022-12-31,...,0.115574,88.746533,NaN,52.949,0.038929,NaN,NaN,NaN,9.0,2026-03-12 11:47:28.083189+00:00
3,GB11429590,Caristo Diagnostics Limited,Active,Baa3,Baa,Take caution,Use caution. Input from outdated and/or partia...,NaN,2022.0,2022-12-31,...,0.000000,35.170227,NaN,73.981,NaN,NaN,NaN,NaN,33.0,2026-03-12 11:47:28.083189+00:00
4,MXCTS160922QLA,Corporate Travel Services Worldwide,Active,B1,B,Do not source,Sourcing exclusion recommended. Mitigation act...,High financial risk,2025.0,2025-03-31,...,14.500631,126.422891,NaN,41.507,0.000000,1.412,2.344,3.641,552.0,2026-03-12 11:47:28.083189+00:00


In [9]:
company_id = df_financials["moodys_bvd_id"].iloc[0]
print(company_id)

df_company = (
    df_financials[
        df_financials["moodys_bvd_id"] == company_id
    ]
    .sort_values(["closing_year", "created_at_utc"])
)

df_company[
    [
        "moodys_bvd_id",
        "closing_year",
        "risk_level",
        "Implied_rating",
        "created_at_utc"
    ]
]

GB03898252


,moodys_bvd_id,closing_year,risk_level,Implied_rating,created_at_utc
13208,GB03898252,2023.0,Take caution,Ba,2026-03-05 09:02:26.421112+00:00
14199,GB03898252,2023.0,Take caution,Ba,2026-03-05 09:02:26.421112+00:00
0,GB03898252,2023.0,Take caution,Ba,2026-03-12 11:47:28.083189+00:00
108,GB03898252,2023.0,Take caution,Ba,2026-03-12 11:47:28.083189+00:00
4396,GB03898252,2023.0,Take caution,Ba,2026-03-12 12:03:33.903081+00:00
4504,GB03898252,2023.0,Take caution,Ba,2026-03-12 12:03:33.903081+00:00
8792,GB03898252,2023.0,Take caution,Ba,2026-03-12 15:07:22.230528+00:00
8900,GB03898252,2023.0,Take caution,Ba,2026-03-12 15:07:22.230528+00:00
14191,GB03898252,2024.0,Take caution,Ba,2026-03-05 09:02:26.421112+00:00
118,GB03898252,2024.0,Take caution,Ba,2026-03-12 11:47:28.083189+00:00


In [10]:
dup_examples = (
    df_financials
    .loc[
        df_financials.duplicated(
            ["moodys_bvd_id", "closing_year"],
            keep=False
        )
    ]
    .sort_values(
        ["moodys_bvd_id", "closing_year", "created_at_utc"]
    )
)

dup_examples.head()

,moodys_bvd_id,moodys_company_name,Status,Implied_rating - original,Implied_rating,risk_level,output_text,Financial_level,closing_year,closing_date,...,Long_term_liabilities_Equity_ratio,Short_term_liabilities_Equity_ratio,Interest_coverage_ratio,Solvency_ratio_Asset_based,Debt_Asset_ratio,ROE_using_Net_income,ROA_using_Net_income,Net_assets_turnover,Number_of_employees,created_at_utc
16565,AE0001327927,Rever Events L.L.C,Active,Ba2,Ba,Take caution,Use caution. Input from outdated and/or partia...,NaN,2024.0,2024-11-26,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03-05 09:02:26.421112+00:00
3508,AE0001327927,Rever Events L.L.C,Active,Ba2,Ba,Take caution,Use caution. Input from outdated and/or partia...,NaN,2024.0,2024-11-26,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03-12 11:47:28.083189+00:00
7904,AE0001327927,Rever Events L.L.C,Active,Ba2,Ba,Take caution,Use caution. Input from outdated and/or partia...,NaN,2024.0,2024-11-26,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03-12 12:03:33.903081+00:00
12300,AE0001327927,Rever Events L.L.C,Active,Ba2,Ba,Take caution,Use caution. Input from outdated and/or partia...,NaN,2024.0,2024-11-26,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03-12 15:07:22.230528+00:00
17219,AE0001327927,Rever Events L.L.C,Active,Ba2,Ba,Take caution,Use caution. Input from outdated and/or partia...,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03-05 09:02:26.421112+00:00


# Check the financial risk distribution 

**"How many suppliers have financial risk?"**

In [11]:
df_unique = df_financials.drop_duplicates(subset="moodys_bvd_id")

distribution_unique = (
    df_unique["Financial_level"]
    .value_counts(dropna=False)
)

print(distribution_unique)

Financial_level
NaN                      1436
Low financial risk        360
Medium financial risk     131
High financial risk        62
Name: count, dtype: int64


In [12]:
risk_distribution = (
    df_financials["Financial_level"]
    .value_counts(dropna=False)
)

print("\nDistribution of Financial Risk:")
print(risk_distribution)


Distribution of Financial Risk:
Financial_level
NaN                      10844
Low financial risk        4266
Medium financial risk     1563
High financial risk        717
Name: count, dtype: int64


In [13]:
risk_percentage = (
    df_financials["Financial_level"]
    .value_counts(normalize=True, dropna=False) * 100
).round(2)

print("\nPercentage distribution:")
print(risk_percentage)


Percentage distribution:
Financial_level
NaN                      62.36
Low financial risk       24.53
Medium financial risk     8.99
High financial risk       4.12
Name: proportion, dtype: float64


`Coverage of financial risk ≈ 37.6%`

`Missing financial risk ≈ 62.4%`

In [14]:
# Create a clean numeric variable for modeling - financial risk score
risk_map = {
    "Low financial risk": 1,
    "Medium financial risk": 2,
    "High financial risk": 3
}

df_financials["financial_risk_score"] = (
    df_financials["Financial_level"]
    .map(risk_map)
)

## Preparing the financial history table

The financial history table contains multiple records for the same company and closing year.  
These repeated rows reflect multiple technical assessment snapshots over time, captured by `created_at_utc`.

For the thesis pipeline, the required structure is:

**one row = one company in one year**

Therefore, the table is cleaned by keeping only the **latest available record** for each combination of:

- `moodys_bvd_id`
- `closing_year`

This preserves the yearly financial history while removing repeated technical snapshots.

The cleaned table will later be joined to the supplier bridge and the contract-year dataset using:

- `moodys_bvd_id`
- `year`

In [15]:
# Work on a copy
df_financials_clean = df_financials.copy()

# Clean key column
df_financials_clean["moodys_bvd_id"] = (
    df_financials_clean["moodys_bvd_id"]
    .astype("string")
    .str.strip()
)

# Parse timestamps
df_financials_clean["created_at_utc"] = pd.to_datetime(
    df_financials_clean["created_at_utc"],
    errors="coerce"
)

df_financials_clean["closing_date"] = pd.to_datetime(
    df_financials_clean["closing_date"],
    errors="coerce"
)

# Remove rows without company id or closing year
df_financials_clean = df_financials_clean[
    df_financials_clean["moodys_bvd_id"].notna() &
    df_financials_clean["closing_year"].notna()
].copy()

# Keep the latest record per company-year
df_financials_clean = (
    df_financials_clean
    .sort_values("created_at_utc")
    .drop_duplicates(
        subset=["moodys_bvd_id", "closing_year"],
        keep="last"
    )
    .copy()
)

# Standardize year column
df_financials_clean["year"] = (
    pd.to_numeric(df_financials_clean["closing_year"], errors="coerce")
    .astype("Int64")
)

# Reorder columns for readability
front_cols = [
    "moodys_bvd_id",
    "moodys_company_name",
    "year",
    "closing_year",
    "closing_date",
    "created_at_utc",
    "Status",
    "Implied_rating",
    "risk_level",
    "Financial_level",
    "output_text",
]

remaining_cols = [col for col in df_financials_clean.columns if col not in front_cols]
df_financials_clean = df_financials_clean[front_cols + remaining_cols]

# Checks
print("Original financial history shape:", df_financials.shape)
print("Clean financial history shape:", df_financials_clean.shape)

print("\nDuplicate company-year rows:",
      df_financials_clean.duplicated(["moodys_bvd_id", "year"]).sum())

print("\nUnique companies:",
      df_financials_clean["moodys_bvd_id"].nunique())

print("\nYear range:")
print(df_financials_clean["year"].min(), "to", df_financials_clean["year"].max())

df_financials_clean.head()

Original financial history shape: (17390, 53)
Clean financial history shape: (3529, 54)

Duplicate company-year rows: 0

Unique companies: 1241

Year range:
1997 to 2026


,moodys_bvd_id,moodys_company_name,year,closing_year,closing_date,created_at_utc,Status,Implied_rating,risk_level,Financial_level,...,Long_term_liabilities_Equity_ratio,Short_term_liabilities_Equity_ratio,Interest_coverage_ratio,Solvency_ratio_Asset_based,Debt_Asset_ratio,ROE_using_Net_income,ROA_using_Net_income,Net_assets_turnover,Number_of_employees,financial_risk_score
14391,GB02490104,Bio-Techne LTD,2022,2022.0,2022-06-30,2026-03-05 09:02:26.421112+00:00,Active,Baa,Take caution,NaN,...,1.348341,29.784142,42.079646,76.259,0.195834,29.641,22.604,0.919,150.0,NaN
14050,DE2110000553,Sartorius AG,2022,2022.0,2022-12-31,2026-03-05 09:02:26.421112+00:00,Active,A,Go ahead,Low financial risk,...,94.603031,67.825040,16.307692,38.106,0.359717,23.651,9.013,0.811,15942.0,1.0
14201,LULB25789,SanisSure S.A.,2021,2021.0,2021-12-31,2026-03-05 09:02:26.421112+00:00,Active,Baa,Take caution,NaN,...,237.538807,34.619388,NaN,26.870,0.000000,NaN,NaN,NaN,NaN,NaN
14532,DE7130049291,Amcor Flexibles Singen GmbH,2022,2022.0,2022-06-30,2026-03-05 09:02:26.421112+00:00,Active,Ba,Take caution,NaN,...,73.990382,309.642979,9.423212,20.677,0.000000,NaN,NaN,4.936,1242.0,NaN
14589,LULB186284,Amazon Web Services EMEA SARL,2022,2022.0,2022-12-31,2026-03-05 09:02:26.421112+00:00,Active,Baa,Go ahead,Low financial risk,...,0.831076,235.324008,0.812609,29.748,0.000000,22.206,6.606,8.358,NaN,1.0


In [16]:
df_financials_clean.duplicated(["moodys_bvd_id", "year"]).sum()

np.int64(0)

Let's look for one example...

In [17]:
company_id = "GB03898252"

raw_company = (
    df_financials[
        df_financials["moodys_bvd_id"] == company_id
    ]
    .sort_values(["closing_year", "created_at_utc"])
)

raw_company[
    [
        "moodys_bvd_id",
        "moodys_company_name",
        "closing_year",
        "created_at_utc",
        "Status",
        "Implied_rating",
        "risk_level",
        "Financial_level",
        "output_text"
    ]
]

,moodys_bvd_id,moodys_company_name,closing_year,created_at_utc,Status,Implied_rating,risk_level,Financial_level,output_text
13208,GB03898252,THE Turner Agency Limited,2023.0,2026-03-05 09:02:26.421112+00:00,Active,Ba,Take caution,NaN,Use caution. Input from outdated and/or partia...
14199,GB03898252,THE Turner Agency Limited,2023.0,2026-03-05 09:02:26.421112+00:00,Active,Ba,Take caution,NaN,Use caution. Input from outdated and/or partia...
0,GB03898252,THE Turner Agency Limited,2023.0,2026-03-12 11:47:28.083189+00:00,Active,Ba,Take caution,NaN,Use caution. Input from outdated and/or partia...
108,GB03898252,THE Turner Agency Limited,2023.0,2026-03-12 11:47:28.083189+00:00,Active,Ba,Take caution,NaN,Use caution. Input from outdated and/or partia...
4396,GB03898252,THE Turner Agency Limited,2023.0,2026-03-12 12:03:33.903081+00:00,Active,Ba,Take caution,NaN,Use caution. Input from outdated and/or partia...
4504,GB03898252,THE Turner Agency Limited,2023.0,2026-03-12 12:03:33.903081+00:00,Active,Ba,Take caution,NaN,Use caution. Input from outdated and/or partia...
8792,GB03898252,THE Turner Agency Limited,2023.0,2026-03-12 15:07:22.230528+00:00,Active,Ba,Take caution,NaN,Use caution. Input from outdated and/or partia...
8900,GB03898252,THE Turner Agency Limited,2023.0,2026-03-12 15:07:22.230528+00:00,Active,Ba,Take caution,NaN,Use caution. Input from outdated and/or partia...
14191,GB03898252,THE Turner Agency Limited,2024.0,2026-03-05 09:02:26.421112+00:00,Active,Ba,Take caution,NaN,Use caution. Input from outdated and/or partia...
118,GB03898252,THE Turner Agency Limited,2024.0,2026-03-12 11:47:28.083189+00:00,Active,Ba,Take caution,NaN,Use caution. Input from outdated and/or partia...


In [18]:
clean_company = (
    df_financials_clean[
        df_financials_clean["moodys_bvd_id"] == company_id
    ]
    .sort_values(["year", "created_at_utc"])
)

clean_company[
    [
        "moodys_bvd_id",
        "moodys_company_name",
        "year",
        "created_at_utc",
        "Status",
        "Implied_rating",
        "risk_level",
        "Financial_level",
        "output_text"
    ]
]

,moodys_bvd_id,moodys_company_name,year,created_at_utc,Status,Implied_rating,risk_level,Financial_level,output_text
8900,GB03898252,THE Turner Agency Limited,2023,2026-03-12 15:07:22.230528+00:00,Active,Ba,Take caution,NaN,Use caution. Input from outdated and/or partia...
8910,GB03898252,THE Turner Agency Limited,2024,2026-03-12 15:07:22.230528+00:00,Active,Ba,Take caution,NaN,Use caution. Input from outdated and/or partia...


In [19]:
print("Raw rows per year:")
print(raw_company["closing_year"].value_counts().sort_index())

print("\nClean rows per year:")
print(clean_company["year"].value_counts().sort_index())

Raw rows per year:
closing_year
2023.0    8
2024.0    4
Name: count, dtype: int64

Clean rows per year:
year
2023    1
2024    1
Name: count, dtype: Int64


In [20]:
df_financials_clean = df_financials_clean.rename(
    columns={"year": "Join_Year"}
)

df_financials_clean["Join_Year"] = pd.to_numeric(
    df_financials_clean["Join_Year"],
    errors="coerce"
).astype("Int64")


In [21]:
print("Total rows:", len(df_financials_clean))

print(
    "Rows with financial risk:",
    df_financials_clean["Financial_level"].notna().sum()
)

print(
    "Rows without financial risk:",
    df_financials_clean["Financial_level"].isna().sum()
)

coverage = (
    df_financials_clean["Financial_level"]
    .notna()
    .mean()
)

print("Coverage:", round(coverage * 100, 1), "%")

Total rows: 3529
Rows with financial risk: 1635
Rows without financial risk: 1894
Coverage: 46.3 %


`Financial risk coverage in financials_clean = 46.3%`

In [22]:
df_financials_clean.head()

,moodys_bvd_id,moodys_company_name,Join_Year,closing_year,closing_date,created_at_utc,Status,Implied_rating,risk_level,Financial_level,...,Long_term_liabilities_Equity_ratio,Short_term_liabilities_Equity_ratio,Interest_coverage_ratio,Solvency_ratio_Asset_based,Debt_Asset_ratio,ROE_using_Net_income,ROA_using_Net_income,Net_assets_turnover,Number_of_employees,financial_risk_score
14391,GB02490104,Bio-Techne LTD,2022,2022.0,2022-06-30,2026-03-05 09:02:26.421112+00:00,Active,Baa,Take caution,NaN,...,1.348341,29.784142,42.079646,76.259,0.195834,29.641,22.604,0.919,150.0,NaN
14050,DE2110000553,Sartorius AG,2022,2022.0,2022-12-31,2026-03-05 09:02:26.421112+00:00,Active,A,Go ahead,Low financial risk,...,94.603031,67.825040,16.307692,38.106,0.359717,23.651,9.013,0.811,15942.0,1.0
14201,LULB25789,SanisSure S.A.,2021,2021.0,2021-12-31,2026-03-05 09:02:26.421112+00:00,Active,Baa,Take caution,NaN,...,237.538807,34.619388,NaN,26.870,0.000000,NaN,NaN,NaN,NaN,NaN
14532,DE7130049291,Amcor Flexibles Singen GmbH,2022,2022.0,2022-06-30,2026-03-05 09:02:26.421112+00:00,Active,Ba,Take caution,NaN,...,73.990382,309.642979,9.423212,20.677,0.000000,NaN,NaN,4.936,1242.0,NaN
14589,LULB186284,Amazon Web Services EMEA SARL,2022,2022.0,2022-12-31,2026-03-05 09:02:26.421112+00:00,Active,Baa,Go ahead,Low financial risk,...,0.831076,235.324008,0.812609,29.748,0.000000,22.206,6.606,8.358,NaN,1.0


In [23]:
print(
    "Duplicate company-year:",
    df_financials_clean.duplicated(
        ["moodys_bvd_id", "Join_Year"]
    ).sum()
)

Duplicate company-year: 0


In [24]:
from master_thesis.config import DATA_RAW

df_financials_clean.to_csv(DATA_RAW / "financials_clean.csv", index=False)
print("Saved to:", DATA_RAW / "financials_clean.csv")


Saved to: /Users/annita/Desktop/Thesis/Signal_Fusion_with_Meta-learners-/Data/raw/financials_clean.csv
